# Упражнение: LightGBM вручную: гистограммы и поиск разбиения по корзинам

## Задание

Дан файл `input.csv`: признаки `x1, x2` и целевая `y` (400 строк). Повторите главный приём LightGBM —
**поиск лучшего разбиения по гистограммам**, а не по всем уникальным значениям.

1. Начальный прогноз F₀ = mean(y). Для потерь ½(F − y)²: градиент gᵢ = F₀ − yᵢ, гессиан hᵢ = 1.
2. Для каждого признака разбейте значения на **8 корзин по квантилям**: границы
   `edges = np.quantile(x, np.linspace(0, 1, 9))` (линейная интерполяция, по умолчанию в NumPy);
   номер корзины = число внутренних границ `edges[1:-1]`, которые ≤ x, т.е. от 0 до 7.
3. Гистограмма признака: для каждой корзины — число объектов `count` и сумма градиентов `grad_sum`.
4. Разбиение «после корзины k» отправляет влево корзины 0…k. Его прирост
   gain = G_L²/(H_L + λ) + G_R²/(H_R + λ) − G²/(H + λ), λ = **1**, где G, H — суммы g и h.
   Для последней корзины (k = 7) разбиения нет — запишите gain = 0.

## Подсказки

- Найти лучшее разбиение можно, не сортируя 400 значений: достаточно накопленных сумм по 8 корзинам.
  Для H при h = 1 сумма гессианов — это просто число объектов.
- Номер корзины: `np.searchsorted(edges[1:-1], x, side='right')`. Гистограмма:
  `np.bincount(bins, weights=g, minlength=8)` и `np.bincount(bins, minlength=8)`.
- G_L для всех k сразу — `np.cumsum(grad_sum)`, H_L — `np.cumsum(count)`; G_R = G − G_L, H_R = H − H_L.
  Проверьте себя: сумма `grad_sum` по корзинам равна нулю (F₀ — среднее), поэтому G²/(H + λ) ≈ 0.

## Ожидаемый формат выходного файла

CSV с колонками `feature, bin, count, grad_sum, gain` — 16 строк: сначала 8 корзин `x1` (bin 0…7),
затем 8 корзин `x2`. `grad_sum` и `gain` округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/lightgbm
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.